# FarmNotary live demo

A research notary for published runs. The simulation stays off-chain.

This notebook runs a tiny consensus-style experiment (stdlib only) and
notarizes it with the **dry-run** backend — no calendars, no pin, no gas.
You should leave with a claim card you can read, not an exit code.

**Missing is not failure.** A printed `Ln` is not scientific correctness.
This notebook will not claim independently reproduced, L0 (no Bitcoin
attestation), or cross-hardware bitwise identity.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "farm_notary").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import farm_notary
from farm_notary import (
    evaluate_claims,
    notarize_run,
    reproduce_run,
    verify_derived_artifacts,
)
from farm_notary.campaign import build_campaign, write_campaign
from farm_notary.diagnose import format_diagnostics
from farm_notary.manifest import (
    build_manifest,
    detect_git_status,
    hash_file,
    is_private_path,
    load_manifest,
)
from farm_notary.paper import build_paper_pack
from farm_notary.precommit import build_precommit, write_precommit
from farm_notary.reproduce import build_receipt, write_receipt
from farm_notary.scope import ALLOWED_SENTENCE

DEMO = ROOT / "docs" / "demo"
EXPERIMENT = DEMO / "experiment.py"
LIVE = DEMO / "_live"
RUN = LIVE / "run-seed-0"
print("repo", ROOT)
print("experiment", EXPERIMENT)
print(f"farm-notary {farm_notary.__version__}")


## 1. Run a tiny official record

Twelve agents vote. The official artifacts are aggregates and a winner.
Individual choices go under `private/` so they must never be hashed.
`scratch_notes.txt` is unmatched on purpose — a forgotten file the
allowlist did not admit.


In [ ]:
if LIVE.exists():
    shutil.rmtree(LIVE)
LIVE.mkdir(parents=True)
subprocess.run(
    [sys.executable, str(EXPERIMENT), "--seed", "0", "--out", str(RUN)],
    check=True,
)
print("run directory:")
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RUN)}")


## 2. A hash tool vs a research notary

`sha256sum` would hash everything, including ballots. FarmNotary hashes
nothing until a profile says what may leave. The denylist still drops
any path containing `ballot`, `vote`, `voter`, `individual_choice`, or
`private`.


In [ ]:
print("would a hash tool see?")
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        rel = path.relative_to(RUN).as_posix()
        flag = "PRIVATE" if is_private_path(rel) else "       "
        print(f"  {flag}  {rel}  {hash_file(path)[:12]}…")

config = json.loads((RUN / "run_config.json").read_text(encoding="utf-8"))
sha, dirty = detect_git_status()
allow_dirty = bool(dirty)
print(f"\ngit {sha[:12] if sha else 'none'}  dirty={dirty}  allow_dirty={allow_dirty}")
print("A dirty tree is refused for a published identity claim. This checkout")
print("opts out only when it is actually dirty.")

probe = build_manifest(
    RUN,
    publish_profile="consensus",
    config=config,
    git_sha=sha,
    git_dirty=dirty,
)
print("\nofficial artifacts:", probe.artifacts)
print("unmatched_count:", probe.unmatched_count)
print("private/ballots.csv in manifest?", "private/ballots.csv" in probe.artifacts)
print("scratch_notes.txt in manifest?", "scratch_notes.txt" in probe.artifacts)


## 3. Stamp the plan before the artifacts exist

A timestamp taken after the result is known is not pre-registration.
`precommit` binds config, command, and code identity *before* the run.
We bind it after this demo run only to show the card row — a lab would
call `precommit` first.


In [ ]:
COMMAND = f"{sys.executable} {EXPERIMENT} --seed 0 --out {{run_dir}}"
precommit = build_precommit(
    config=config,
    command=COMMAND,
    git_sha=sha,
    git_dirty=dirty,
    allow_dirty=allow_dirty,
)
precommit_path = LIVE / "precommit.json"
write_precommit(precommit, precommit_path)
print("precommit schema", precommit["schema"])
print("command", precommit["command"])
print("wrote", precommit_path)


## 4. Notarize and read the card

`notarize_run` writes `manifest.json` and a dry-run anchor. Reviewers
read the card, not the exit code. Last row is always
`not claimed: scientific correctness`.


In [ ]:
manifest, receipt = notarize_run(
    RUN,
    publish_profile="consensus",
    config=config,
    git_sha=sha,
    git_dirty=dirty,
    command=COMMAND,
    runner="demo_consensus",
    precommit_path=precommit_path,
    allow_dirty=allow_dirty,
)
print("content_hash", manifest.content_hash())
print("anchor backend", receipt.backend, "dry_run", receipt.dry_run)
print()
card = evaluate_claims(manifest, RUN)
print(card.render(), end="")
print("card.ok (no attempted check failed):", card.ok)


## 5. Tamper is visible

Change one official byte. Verify must fail. Restore the file so the
rest of the demo can continue.


In [ ]:
summary = RUN / "summary.csv"
original = summary.read_bytes()
summary.write_bytes(original + b"\n# tampered\n")
tampered = evaluate_claims(load_manifest(RUN), RUN)
print(tampered.render(), end="")
print("problems:")
for problem in tampered.problems:
    print(" ", problem)
summary.write_bytes(original)
restored = evaluate_claims(load_manifest(RUN), RUN)
print("\nrestored. card.ok:", restored.ok)
if not restored.ok:
    print(restored.problems)


## 6. Re-run the seed. Scope the sentence.

`reproduce` executes the recorded command into a fresh directory and
byte-compares every listed artifact. A passing receipt on this machine
may emit the only sentence the tool is allowed today:

> byte-identical on x86-64 Linux in a pinned environment


In [ ]:
print(ALLOWED_SENTENCE)
fresh = LIVE / "repro-seed-0"
result = reproduce_run(manifest, fresh_dir=fresh, original_dir=RUN)
print("\n".join(result.summary()))
print("ok", result.ok, "matched", result.matched)
receipt_doc = build_receipt(manifest, result)
write_receipt(receipt_doc, RUN)
card = evaluate_claims(load_manifest(RUN), RUN)
print()
print(card.render(), end="")


## 7. A byte-diff is not a science failure

Embed the output path in `REPORT.md` — the packaging bug that took the
AgentFarm consensus experiment from 7/7 to 8/8. FarmNotary classifies
it. The science did not change.


In [ ]:
broken = LIVE / "run-embedded"
subprocess.run(
    [sys.executable, str(EXPERIMENT), "--seed", "0", "--out", str(broken), "--embed-path"],
    check=True,
)
broken_manifest, _ = notarize_run(
    broken,
    publish_profile="consensus",
    config=json.loads((broken / "run_config.json").read_text(encoding="utf-8")),
    git_sha=sha,
    git_dirty=dirty,
    command=f"{sys.executable} {EXPERIMENT} --seed 0 --out {{run_dir}} --embed-path",
    runner="demo_consensus",
    allow_dirty=allow_dirty,
)
broken_result = reproduce_run(
    broken_manifest,
    fresh_dir=LIVE / "repro-embedded",
    original_dir=broken,
)
print("\n".join(broken_result.summary()))
print()
print("\n".join(format_diagnostics(broken_result.diagnostics)))


## 8. Statistics recompute — only if you trust the commands

Default `verify` does not execute `derived_from` on a downloaded
manifest. Pass the opt-in for a record you wrote.


In [ ]:
notes = verify_derived_artifacts(manifest, RUN, allow_execute=False)
print("without opt-in:")
for line in notes:
    print(" ", line)
problems = verify_derived_artifacts(manifest, RUN, allow_execute=True)
print("\nwith allow_execute=True:")
print(" ", "pass" if not problems else problems)


## 9. The figure is a parent record, not a folder

Two seeds, one campaign, one appendix snippet a paper can paste.
Reader ladder is an em dash: do not cite `Ln` from the PDF.


In [ ]:
run1 = LIVE / "run-seed-1"
subprocess.run(
    [sys.executable, str(EXPERIMENT), "--seed", "1", "--out", str(run1)],
    check=True,
)
cfg1 = json.loads((run1 / "run_config.json").read_text(encoding="utf-8"))
notarize_run(
    run1,
    publish_profile="consensus",
    config=cfg1,
    git_sha=sha,
    git_dirty=dirty,
    command=f"{sys.executable} {EXPERIMENT} --seed 1 --out {{run_dir}}",
    runner="demo_consensus",
    allow_dirty=allow_dirty,
)
campaign = build_campaign([RUN, run1], name="demo-sweep", campaign_dir=LIVE / "sweep")
write_campaign(campaign, LIVE / "sweep")
print("children", [(row.get("seed"), row["content_hash"][:12]) for row in campaign.runs])
print()
print(build_paper_pack(load_manifest(RUN), RUN, experiment="demo_consensus"))


## What you may take with you

You produced a tamper-evident official record, bound a precommit, re-ran
the seed, and saw a packaging mismatch labeled **not a science failure**.

You may **not** take: a verified result, L0 (no Bitcoin attestation),
independently reproduced, or cross-hardware bitwise identity. Immutability
is not correctness.

Next, for a run you would cite: `farm-notary anchor --backend ots --pin-remote pinata`,
then `farm-notary upgrade` once a calendar has a Bitcoin height.
